Copyright 2021 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required bsteps_per_epoch y applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.steps_per_epoch 
# limitations under the License.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/privacy/blob/master/tensorflow_privacy/privacy/privacy_tests/secret_sharer/secret_sharer_example.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

#Setup

You may set the runtime to use a GPU by Runtime > Change runtime type > Hardware accelerator.

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [ ]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [ ]:


SEQ_LENGTH = 1024

BATCHES_PER_EPOCH = 200
TRAIN_EPOCHS = 20
BATCH_SIZE = 16
DATASET_SIZE = 20000




In [ ]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    """Carga el texto de la columna 'TEXT' de un archivo CSV."""
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().iloc[:DATASET_SIZE].tolist()) 
    return text

In [ ]:
#@title Function for getting dataset combined with secrets.
def get_dataset(secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv("MIMIC_NoteEvents_deanonymized.csv")

  # Create vocab
  vocab = sorted(set(text))
  char2idx = {u: i for i, u in enumerate(vocab)}
  idx2char = np.array(vocab)

  # Gather all secrets
  if not secrets:
    pass
  else:
    all_secrets = []
    for secret in secrets:
      secret.datasets = {}
      for r, seqs in secret.secrets.items():
        all_secrets += seqs * r
        indexed_seqs = np.array([tokenizer.encode(s, padding='max_length', max_length=seq_length) for s in seqs])
        secret.datasets[r] = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
            'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
        }
      indexed_seqs = np.array([tokenizer.encode(s,padding='max_length', max_length=seq_length) for s in secret.references])
      secret.datasets[0] = {
          'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
          #'label': list(map(lambda chunk: chunk[1:], indexed_seqs))
          'label': list(map(lambda chunk: np.roll(chunk, -1), indexed_seqs))
      }
    # Insert the secrets into the training data
    idx = np.random.choice(len(text) - 1, len(all_secrets) - 1, replace=False) + 1
    # split text into len(all_secrets) parts
    text = np.split(list(text), np.sort(idx))
    text = [''.join(t) for t in text]
    text = ''.join(np.vstack([text, all_secrets]).transpose().reshape(-1))

  # Split text into sequence length + 1 chucks to create examples
  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 usa una secuencia máxima de 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(10000).batch(8, drop_remainder=True)  # Batch final
    
  #text_as_int = np.array([char2idx[c] for c in text])
  #char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
  #sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

  #dataset = sequences.map(split_input_target)
  #dataset = dataset.shuffle(10000).repeat()
  #dataset = dataset.batch(batch_size, drop_remainder=True)

  return dataset, idx2char, char2idx

In [ ]:
from tensorflow_privacy.privacy.analysis import compute_noise_from_budget

epsilon = 10
delta = 1 / DATASET_SIZE
l2_norm_clip = 1.0 
# Calculamos el noise_multiplier para garantizar (ε, δ)-DP
noise_multiplier = compute_noise_from_budget(
    n=DATASET_SIZE,  # Tamaño del dataset
    batch_size=BATCH_SIZE,
    epsilon=epsilon,
    epochs=train_epochs,
    delta=delta,
    min_noise=1e-6
)

In [ ]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer

def train_model(dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Entrena un modelo Transformer con checkpoints."""

    train_steps = BATCHES_PER_EPOCH // batch_size 

    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 0.4272711315437153
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    # Callback para guardar checkpoints
    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Guarda solo el mejor modelo
            monitor="loss",  # Basado en la pérdida
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)

        # Restaurar último checkpoint si existe
        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'✅ Checkpoint encontrado: {checkpoint_latest}. Continuando desde la época {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0

    # Callback para medir el tiempo de cada época (opcional)
    time_callback = tf.keras.callbacks.TensorBoard(log_dir="./logs", histogram_freq=1)
    callbacks.append(time_callback)

    # Entrenar el modelo
    history = model.fit(
        dataset,
        epochs=train_epochs,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
        callbacks=callbacks,
        verbose=2
    )

    return history, callbacks

In [ ]:
from transformers import TFAutoModelForCausalLM
import tensorflow as tf

def load_gpt2_model(model_checkpoint, checkpoint_dir, dp):
    """Carga un modelo GPT-2 con pesos desde el último checkpoint."""
    
    # Construir el modelo GPT-2
    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)

    
    if dp:
        noise_multiplier = 0.4272711315437153
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("🔒 Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)
    
    # Cargar los pesos del último checkpoint si existe
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'✅ Pesos cargados desde {latest_checkpoint}')
    else:
        print('⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.')

    # Compilar el modelo con la pérdida adecuada para modelos de lenguaje
    prediction_model.compile(optimizer = optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [84]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf

def compute_perplexity_for_secret(secrets, checkpoint_dir, vocab_size, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size

    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = [{} for _ in range(len(secrets))]
    ppls_reference = [None] * len(secrets)
    
    for i, secret in enumerate(secrets):
        print("i:", i)
        for r in secret.datasets.keys():
            d = secret.datasets[r]
            indexed_seqs = d['data']  # This is the _BatchDataset
            all_losses = []
            # Iterate over the dataset (which is a batch of sequences)
            for seq_batch in indexed_seqs:
                # seq_batch is a batch of sequences, iterate over each sequence in the batch
                for seq in seq_batch:
                    tokens = tf.convert_to_tensor([seq])  # Convert the sequence to a tensor and add batch dimension
                    
                    # Get model logits for the sequence
                    logits = prediction_model.predict(tokens).logits

                    # Apply softmax to get probabilities
                    predictions_prob = softmax(logits, axis=-1)
                
                    # Calculate the loss (log-probabilities) for each token in the sequence
                    loss = 0
                    for k in range(1, len(seq)):  # Skip the first token, as it's the start token
                        token = seq[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  # Sum of log2 probabilities

                    # Store the loss for the current sequence
                    all_losses.append(loss / len(seq))  # Average loss over the sequence length

            # Store the perplexity for this dataset
            print("LOSS:", all_losses)
            if r == 0:
                ppls_reference[i] = all_losses
            else:
                ppls[i][r] = all_losses

    return ppls, ppls_reference


# Run training and secret sharer on Shakespeare dataset.

In [ ]:
#@title Generate secrets
# Here we present just one example and you can try other combinations of patterns,
# vocabs and number of repetitions. For example, you may use other patterns and
# different vocabularies for them.
patterns = ['{}'*5,
            'Your doctor is ' + '{}'*5,
            '{}{}-{}{}-{}',
            '{}' * 10]
vocabs = [
    ["Abel", "Adam", "Carl", "Dean", "Gill", "Grey", "Hunt", "Leon", "Lynn", "Reed"]
] * 4
num_repetitions = [100, 1000, 5000]  #CAMBIAR CANT REPETICIONES
num_secrets_for_repetitions = [30] * len(num_repetitions)
num_references = 25 #65536
secret_configs = [SecretConfig(num_repetitions, num_secrets_for_repetitions, num_references,
                               properties=TextSecretProperties(vocab, pattern))
                  for vocab, pattern in zip(vocabs, patterns)]
secrets = generate_text_secrets_and_references(secret_configs)

# Let's look at the variable "secrets"
print(f'"secrets" is a list and the length is {len(secrets)} because we have four sets of secrets.')
print(f'The type of the element in "secrets" is {type(secrets[0])}.')
print(f'The field "config" contains its configuration: {secrets[0].config}')
print(f'The filed "secrets" is a dictionary, with key being the number of repetitions and value being a list of secrets.',
      f'For example, the secrets that are going to be repeated for 100 times are {secrets[0].secrets[100]}.')
print(f'The field "references" is a list of reference sequences that are not going to be injected. The first 10 elements are {secrets[0].references[:10]}.')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
for i in range(len(secrets)):
  dataset, idx2char, char2idx = get_dataset(
      [secrets[i]],
      batch_size=BATCH_SIZE)
    
  checkpoint_dir = 'DP-checkpoints'+str(i)
  print("dataset", i, "loaded")
  history, callbacks = train_model(dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

In [75]:
#@title Secret sharer evaluation
ppls_all, ppls_reference_all = [], []
for i in range(len(secrets)):
  checkpoint_dir =  'checkpoints'+str(i)
  ppls, ppls_reference = compute_perplexity_for_secret(
      [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
  ppls_all += ppls
  ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  print("p", p, "p_r",p_r)
  exposure_interpolation = compute_exposure_interpolation(p, p_r)

  exposure_extrapolation = compute_exposure_extrapolation(p, p_r)

  # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
  # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
  # the average exposure (with standard deviation).
  print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
  print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
  print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.
i: 0
1/1 [==============================] - 1s 586ms/step
LOSS: [16.858700785002984, 16.89646229809214, 16.866055924449032, 16.905258040157815, 16.88592081055963, 16.83063572219856, 16.859276970227558, 16.78745671404655, 16.905514316241877, 16.83769840205986, 16.87615945378823, 16.82765805814506, 16.847791673850455, 16.792192093146745, 16.88612349047339, 16.915595257154074, 16.811448593409995, 16.842335700173066, 16.802532979353664, 16.860885299662108, 16.88803099443952, 16.96846145851754, 16.844214830230758, 16.837963001818952, 16.825281183041437, 16.906684597770255, 16.83574957978108, 16.886160988145793, 16.88297186825399, 16.76985047849392]


IndexError: list index out of range

# Comparing models

In [76]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

dataset, idx2char, char2idx = get_dataset(
  [secrets[1]],
  batch_size=BATCH_SIZE)

checkpoint_dir = 'DP-checkpoints'
history, callbacks = train_model(dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

checkpoint_dir = 'checkpoints'
history, callbacks = train_model(dataset, batch_size=32, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=False)

Token indices sequence length is longer than the specified maximum sequence length for this model (2173099 > 1024). Running this sequence through the model will result in indexing errors
All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


🔒 Usando DP-SGD con noise_multiplier = 0.4272711315437153
✅ Checkpoint encontrado: DP-checkpoints/ckpt_1. Continuando desde la época 1.
Epoch 2/20

Epoch 2: loss improved from inf to 3.24969, saving model to DP-checkpoints/ckpt_2
6/6 - 270s - loss: 3.2497 - 270s/epoch - 45s/step
Epoch 3/20

Epoch 3: loss improved from 3.24969 to 3.24045, saving model to DP-checkpoints/ckpt_3
6/6 - 212s - loss: 3.2405 - 212s/epoch - 35s/step
Epoch 4/20

Epoch 4: loss improved from 3.24045 to 3.21635, saving model to DP-checkpoints/ckpt_4
6/6 - 210s - loss: 3.2163 - 210s/epoch - 35s/step
Epoch 5/20

Epoch 5: loss improved from 3.21635 to 3.19391, saving model to DP-checkpoints/ckpt_5
6/6 - 210s - loss: 3.1939 - 210s/epoch - 35s/step
Epoch 6/20

Epoch 6: loss improved from 3.19391 to 3.18497, saving model to DP-checkpoints/ckpt_6
6/6 - 211s - loss: 3.1850 - 211s/epoch - 35s/step
Epoch 7/20

Epoch 7: loss did not improve from 3.18497
6/6 - 211s - loss: 3.1966 - 211s/epoch - 35s/step
Epoch 8/20

Epoch 8: lo

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Epoch 1/20

Epoch 1: loss improved from inf to 25.25494, saving model to checkpoints/ckpt_1
6/6 - 159s - loss: 25.2549 - 159s/epoch - 27s/step
Epoch 2/20

Epoch 2: loss did not improve from 25.25494
6/6 - 134s - loss: 33.4115 - 134s/epoch - 22s/step
Epoch 3/20

Epoch 3: loss improved from 25.25494 to 10.20034, saving model to checkpoints/ckpt_3
6/6 - 135s - loss: 10.2003 - 135s/epoch - 22s/step
Epoch 4/20

Epoch 4: loss did not improve from 10.20034
6/6 - 134s - loss: 16.4787 - 134s/epoch - 22s/step
Epoch 5/20

Epoch 5: loss did not improve from 10.20034
6/6 - 134s - loss: 11.5572 - 134s/epoch - 22s/step
Epoch 6/20

Epoch 6: loss improved from 10.20034 to 7.02046, saving model to checkpoints/ckpt_6
6/6 - 135s - loss: 7.0205 - 135s/epoch - 23s/step
Epoch 7/20

Epoch 7: loss improved from 7.02046 to 6.29069, saving model to checkpoints/ckpt_7
6/6 - 135s - loss: 6.2907 - 135s/epoch - 23s/step
Epoch 8/20

Epoch 8: loss improved from 6.29069 to 5.99019, saving model to checkpoints/ckpt_8
6/

In [85]:
#@title Comparing exposure between base model, DP-SGD and SGD

#SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

#DP-SGD model:
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'DP-checkpoints'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size, dp=True)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))
#base model (GPT2 without finetuning):
ppls_all, ppls_reference_all = [], []
checkpoint_dir =  'base'
ppls, ppls_reference = compute_perplexity_for_secret(
  [secrets[1]], checkpoint_dir, tokenizer.vocab_size)
ppls_all += ppls
ppls_reference_all += ppls_reference


for i, (vocab, pattern, p, p_r) in enumerate(zip(vocabs, patterns, ppls_all, ppls_reference_all)):
  if (i != 0):
      break
  else:
      exposure_interpolation = compute_exposure_interpolation(p, p_r)
    
      exposure_extrapolation = compute_exposure_extrapolation(p, p_r)
    
      # exposure_interpolation (or exposure_extrapolation) is a dictionary, where the key is the number of repetitions and
      # the value is a list of exposures (not necessarily in the same order as the secrets). We will now take a look at
      # the average exposure (with standard deviation).
      print(f'Secrets #{i+1}, vocab={vocab}, pattern={pattern}')
      print('  Interpolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_interpolation[r]):.2f}±{np.std(exposure_interpolation[r]):.2f}' for r in num_repetitions]))
      print('  Extrapolation:', '; '.join([f'repetition={r}, avg_exposure={np.mean(exposure_extrapolation[r]):.2f}±{np.std(exposure_extrapolation[r]):.2f}' for r in num_repetitions]))

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


✅ Pesos cargados desde checkpoints/ckpt_17
i: 0
1/1 [==============================] - 1s 550ms/step
LOSS: [12.978184221310583, 12.97090820296652, 12.971675931999528, 12.972453050949007, 12.972925749458991, 12.989047224570578, 12.979736186658648, 12.976238254228296, 12.972263894123067, 12.99047754377214, 12.979968029965637, 12.983907205035376, 12.972580252970884, 12.97639352182256, 12.976038849365443, 12.987613013762532, 12.977632224734577, 12.98850889895785, 12.984189495429732, 12.988580095919463, 12.982445083871731, 12.961299115961248, 12.991053492209545, 12.976152852833096, 12.970930082240995, 12.98140436463342, 12.985697217933826, 12.969750828290849, 12.978946163274442, 12.989088309708462]
i VALE 0
1/1 [==============================] - 1s 599ms/step
LOSS: [12.97823751333988, 12.964449913271013, 12.98430319895143, 12.9693625889216, 12.978812594334517, 12.977613065482817, 12.972132627332432, 12.979973842904249, 12.968103020072332, 12.970091957267545, 12.970958646091898, 12.970670705

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


🔒 Usando DP-SGD con noise_multiplier = 0.4272711315437153
✅ Pesos cargados desde DP-checkpoints/ckpt_13
i: 0
1/1 [==============================] - 1s 622ms/step
LOSS: [16.858671617065124, 16.896433947955643, 16.866028006577306, 16.90522993374197, 16.885890500403335, 16.830604906772006, 16.859248561127327, 16.787427168321283, 16.905485949791306, 16.837667429085933, 16.87612948715978, 16.827627748804474, 16.847762976690237, 16.792163324612442, 16.886094887352176, 16.915566680951084, 16.811419550158295, 16.84230654633523, 16.802503657830425, 16.860855642535004, 16.88800137506785, 16.96843329727475, 16.844184618425043, 16.83793306123476, 16.82525226313581, 16.906655859795716, 16.835721466431753, 16.886132386248132, 16.882942870681703, 16.769822230903053]
i VALE 0
1/1 [==============================] - 1s 592ms/step
LOSS: [16.86686220360175, 16.92943044454005, 16.84542732457844, 16.961709579298105, 16.911288722169715, 16.832763971354836, 16.902160542102155, 16.881347403847926, 16.932316846

All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


⚠️ No se encontró ningún checkpoint, cargando pesos preentrenados.
i: 0
1/1 [==============================] - 1s 564ms/step
LOSS: [16.858700785002984, 16.89646229809214, 16.866055924449032, 16.905258040157815, 16.88592081055963, 16.83063572219856, 16.859276970227558, 16.78745671404655, 16.905514316241877, 16.83769840205986, 16.87615945378823, 16.82765805814506, 16.847791673850455, 16.792192093146745, 16.88612349047339, 16.915595257154074, 16.811448593409995, 16.842335700173066, 16.802532979353664, 16.860885299662108, 16.88803099443952, 16.96846145851754, 16.844214830230758, 16.837963001818952, 16.825281183041437, 16.906684597770255, 16.83574957978108, 16.886160988145793, 16.88297186825399, 16.76985047849392]
i VALE 0
1/1 [==============================] - 1s 613ms/step
LOSS: [16.86689148643732, 16.929458654302657, 16.84545689920293, 16.961739326385693, 16.911317622789774, 16.832794128624926, 16.902189991399574, 16.881376225228184, 16.93234502931261, 16.93000788026774, 16.8904695517856

We can see the following:
1. Comparing #1 vs. #2, having the prefix (#2) for the random sequences might help the model memorize.
2. Comparing #1 vs. #3, having the pattern (#3) did not help the memorization significantly. This is different from the result in the [paper](https://arxiv.org/abs/1802.08232). The effect might be model / data specific.
3. Comparing #1 vs. #4, longer secrets (#4) show higher exposure.

You can also try some other secret configurations to see if there is any difference.